# CPC TorchRL Scaffold

This notebook is the initiating experiment entry point. Core logic lives in Python modules; the notebook demonstrates the toy CPC loop and the thin TorchRL wrapper.

In [ ]:
from pathlib import Path
import json
import sys

EXPERIMENT_ROOT = Path.cwd()
if EXPERIMENT_ROOT.name != "experiment":
    EXPERIMENT_ROOT = Path.cwd() / "experiment"

if str(EXPERIMENT_ROOT) not in sys.path:
    sys.path.insert(0, str(EXPERIMENT_ROOT))

from cpc_actions import action_space, decode_action, describe_action
from cpc_env import CPCEnv

print("experiment root:", EXPERIMENT_ROOT)
print("action space:", action_space())

## PR1: Toy CPC Loop

In [ ]:
env = CPCEnv(seed=7, max_steps=8)
obs = env.reset()
obs

## Decode Actions

`move`, `aim`, and `fire` are separate control channels. This means movement and firing can happen in the same step.

In [ ]:
move_and_fire = {"move": 6, "aim": 0, "fire": 1}  # up_right + aim east + fire
print(json.dumps(describe_action(move_and_fire), indent=2))

decoded = decode_action(move_and_fire)
assert decoded["moveX"] > 0
assert decoded["moveY"] < 0
assert decoded["fire"] == 1

## Step With Move + Fire

In [ ]:
obs, reward, done, info = env.step(move_and_fire)
print("observation")
print(json.dumps(obs, indent=2))
print("reward", reward, "done", done)
print("decoded action")
print(json.dumps(info["decoded_action"], indent=2))
print("reward components")
print(json.dumps(info["reward_components"], indent=2))

## Run Random Actions

In [ ]:
for _ in range(5):
    action = env.sample_action()
    obs, reward, done, info = env.step(action)
    print({
        "raw": action,
        "decoded": info["decoded_action"],
        "reward": round(reward, 3),
        "done": done,
        "distance_to_ally": round(obs["distance_to_ally"], 2),
    })
    if done:
        break

## Metrics And Trajectory Export

In [ ]:
metrics = env.metrics.summary()
trajectory = env.export_trajectory()

print("metrics")
print(json.dumps(metrics, indent=2))

print("sample trajectory step")
print(json.dumps(trajectory["steps"][0], indent=2))

# PR2: TorchRL EnvBase / TensorDict Wrapper

The wrapper adapts the toy env to TorchRL with flat action keys: `move`, `aim`, and `fire`. PPO is intentionally left for PR3.

In [ ]:
try:
    import torch
    from torchrl_env import TorchRLCPCEnv
    from torchrl_specs import import_check_env_specs

    torchrl_env = TorchRLCPCEnv(seed=11, max_steps=8)
    print("observation_spec")
    print(torchrl_env.observation_spec)
    print("action_spec")
    print(torchrl_env.action_spec)

    td = torchrl_env.reset()
    print("reset keys", td.keys())

    action_td = torchrl_env.action_spec.rand()
    td.update(action_td)
    stepped = torchrl_env.step(td)
    print("sampled action", action_td)
    print("reward", stepped["next", "reward"], "done", stepped["next", "done"])

    rollout_td = torchrl_env.reset()
    for _ in range(5):
        rollout_td.update(torchrl_env.action_spec.rand())
        rollout_td = torchrl_env.step(rollout_td)["next"]
        print("random step", rollout_td["step_count"].item(), rollout_td["reward"].item(), rollout_td["done"].item())
        if rollout_td["done"].item():
            break

    move_fire_td = torchrl_env.reset()
    move_fire_td["move"] = torch.tensor(6, dtype=torch.int64)  # up_right
    move_fire_td["aim"] = torch.tensor(0, dtype=torch.int64)
    move_fire_td["fire"] = torch.tensor(1, dtype=torch.int64)
    move_fire_next = torchrl_env.step(move_fire_td)
    print("move+fire reward", move_fire_next["next", "reward"])
    print("move+fire done", move_fire_next["next", "done"])
    print("decoded engine action", move_fire_next["next", "decoded_action"])

    check_env_specs = import_check_env_specs()
    if check_env_specs is not None:
        check_env_specs(torchrl_env)
        print("check_env_specs passed")
    else:
        print("check_env_specs unavailable in this TorchRL version")
except Exception as exc:
    print("TorchRL wrapper demo skipped:", exc)

# TODO PR3: add PPO actor-critic heads for move, aim, and fire.
# TODO PR3: add collector, PPO loss, checkpointing, and smoke training.